In [35]:
from pathlib import Path

project = Path(r"D:\ECommerce_ETL_Project")

print("Project folder exists:", project.exists())
print("\nFiles and folders inside project:")

if project.exists():
    for item in project.iterdir():
        print(item.name, "— folder" if item.is_dir() else "— file")

print("\nFiles inside data:")

data_folder = project / "data"

if data_folder.exists():
    for item in data_folder.rglob("*"):
        print(item)

Project folder exists: True

Files and folders inside project:
data — folder
documentation — folder
notebooks — folder
output — folder
sql — folder

Files inside data:
D:\ECommerce_ETL_Project\data\cleaned
D:\ECommerce_ETL_Project\data\raw
D:\ECommerce_ETL_Project\data\rejected
D:\ECommerce_ETL_Project\data\cleaned\clickstream_clean
D:\ECommerce_ETL_Project\data\raw\.ipynb_checkpoints
D:\ECommerce_ETL_Project\data\raw\clickstream.json
D:\ECommerce_ETL_Project\data\raw\customers.csv
D:\ECommerce_ETL_Project\data\raw\products.csv
D:\ECommerce_ETL_Project\data\raw\sales.csv
D:\ECommerce_ETL_Project\data\raw\.ipynb_checkpoints\customers-checkpoint.csv
D:\ECommerce_ETL_Project\data\raw\.ipynb_checkpoints\products-checkpoint.csv
D:\ECommerce_ETL_Project\data\raw\.ipynb_checkpoints\sales-checkpoint.csv


In [36]:
from pathlib import Path

RAW_PATH = Path(r"D:\ECommerce_ETL_Project\data\raw")

print("Clickstream exists:", (RAW_PATH / "clickstream.json").exists())
print("Sales exists:", (RAW_PATH / "sales.csv").exists())

Clickstream exists: True
Sales exists: True


In [37]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("ECommerce_MultiChannel_ETL")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.2.0


In [38]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [39]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)

clickstream_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category_id", StringType(), True),
    StructField("price", DoubleType(), True)
])

sales_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("transaction_id", StringType(), True),
    StructField("date_time", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("product_id", StringType(), True),
    StructField("status", StringType(), True)
])

print("Both schemas created successfully.")

Both schemas created successfully.


In [40]:
clickstream_raw = (
    spark.read
    .schema(clickstream_schema)
    .option("mode", "PERMISSIVE")
    .json(str(CLICKSTREAM_PATH))
)

sales_raw = (
    spark.read
    .schema(sales_schema)
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .csv(str(SALES_PATH))
)

print("Clickstream records:", clickstream_raw.count())
print("Retail sales records:", sales_raw.count())

clickstream_raw.show(5, truncate=False)
sales_raw.show(5, truncate=False)

Clickstream records: 30
Retail sales records: 23
+-------+----------+----------+-------------------+----------+-----------+-----+
|user_id|session_id|event_type|event_time         |product_id|category_id|price|
+-------+----------+----------+-------------------+----------+-----------+-----+
|101    |S001      |view      |2026-10-01 10:15:00|P101      |C01        |500.0|
|101    |S001      |cart      |2026-10-01 10:18:00|P101      |C01        |500.0|
|101    |S001      |purchase  |2026-10-01 10:22:00|P101      |C01        |500.0|
|102    |S002      |view      |2026-10-01 10:25:00|P102      |C02        |800.0|
|102    |S002      |cart      |2026-10-01 10:31:00|P102      |C02        |800.0|
+-------+----------+----------+-------------------+----------+-----------+-----+
only showing top 5 rows
+-----------+--------------+-------------------+--------+------+----------+---------+
|customer_id|transaction_id|date_time          |quantity|price |product_id|status   |
+-----------+-------------

In [41]:
## cleaned chickstream data
from pyspark.sql.functions import col, try_to_timestamp
from pyspark.sql.functions import lit, try_to_timestamp
clickstream_parsed = clickstream_raw.withColumn(
    "event_time",
    try_to_timestamp(
        col("event_time"),
        lit("yyyy-MM-dd HH:mm:ss")
    )
)

clickstream_valid = clickstream_parsed.filter(
    col("user_id").isNotNull() &
    (col("user_id") != "") &
    col("session_id").isNotNull() &
    (col("session_id") != "") &
    col("event_time").isNotNull() &
    col("event_type").isin("view", "cart", "purchase") &
    col("price").isNotNull() &
    (col("price") > 0)
)

clickstream_clean = clickstream_valid.dropDuplicates([
    "user_id", "session_id", "event_type", "event_time", "product_id"
])

print("Original clickstream records:", clickstream_raw.count())
print("Clean clickstream records:", clickstream_clean.count())
print("Rejected clickstream records:",
      clickstream_raw.count() - clickstream_clean.count())

clickstream_clean.show(10, truncate=False)

Original clickstream records: 30
Clean clickstream records: 26
Rejected clickstream records: 4
+-------+----------+----------+-------------------+----------+-----------+------+
|user_id|session_id|event_type|event_time         |product_id|category_id|price |
+-------+----------+----------+-------------------+----------+-----------+------+
|101    |S001      |view      |2026-10-01 10:15:00|P101      |C01        |500.0 |
|101    |S001      |cart      |2026-10-01 10:18:00|P101      |C01        |500.0 |
|101    |S001      |purchase  |2026-10-01 10:22:00|P101      |C01        |500.0 |
|102    |S002      |view      |2026-10-01 10:25:00|P102      |C02        |800.0 |
|102    |S002      |cart      |2026-10-01 10:31:00|P102      |C02        |800.0 |
|102    |S002      |purchase  |2026-10-01 10:35:00|P102      |C02        |800.0 |
|103    |S003      |view      |2026-10-01 11:05:00|P103      |C01        |300.0 |
|103    |S003      |cart      |2026-10-01 11:10:00|P103      |C01        |300.0 |
|10

In [43]:
##Clean sales data

from pyspark.sql.functions import col, lit, try_to_timestamp

sales_parsed = (
    sales_raw
    .withColumn(
        "transaction_time",
        try_to_timestamp(
            col("date_time"),
            lit("yyyy-MM-dd HH:mm:ss")
        )
    )
    .withColumn(
        "total_amount",
        col("quantity") * col("price")
    )
)

sales_valid = sales_parsed.filter(
    col("customer_id").isNotNull() &
    (col("customer_id") != "") &
    col("transaction_id").isNotNull() &
    (col("transaction_id") != "") &
    col("transaction_time").isNotNull() &
    col("quantity").isNotNull() &
    (col("quantity") > 0) &
    col("price").isNotNull() &
    (col("price") > 0) &
    (col("status") == "FINALIZED")
)

sales_clean = sales_valid.dropDuplicates(["transaction_id"])

print("Original sales records:", sales_raw.count())
print("Clean sales records:", sales_clean.count())
print(
    "Rejected sales records:",
    sales_raw.count() - sales_clean.count()
)

sales_clean.select(
    "customer_id",
    "transaction_id",
    "transaction_time",
    "quantity",
    "price",
    "total_amount",
    "product_id",
    "status"
).show(20, truncate=False)

Original sales records: 23
Clean sales records: 16
Rejected sales records: 7
+-----------+--------------+-------------------+--------+------+------------+----------+---------+
|customer_id|transaction_id|transaction_time   |quantity|price |total_amount|product_id|status   |
+-----------+--------------+-------------------+--------+------+------------+----------+---------+
|101        |T001          |2026-10-01 10:22:00|1       |500.0 |500.0       |P101      |FINALIZED|
|102        |T002          |2026-10-01 10:35:00|2       |800.0 |1600.0      |P102      |FINALIZED|
|103        |T003          |2026-10-01 11:18:00|1       |300.0 |300.0       |P103      |FINALIZED|
|105        |T005          |2026-10-01 12:12:00|1       |450.0 |450.0       |P105      |FINALIZED|
|106        |T006          |2026-10-01 12:25:00|1       |1500.0|1500.0      |P106      |FINALIZED|
|107        |T007          |2026-10-01 13:11:00|1       |700.0 |700.0       |P107      |FINALIZED|
|109        |T009          |2026

In [44]:
## save cleanded data

import os

OUTPUT_PATH = PROJECT_ROOT / "output"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

clickstream_clean.toPandas().to_csv(
    str(OUTPUT_PATH / "clickstream_clean.csv"),
    index=False
)

sales_clean.toPandas().to_csv(
    str(OUTPUT_PATH / "sales_clean.csv"),
    index=False
)

print("Cleaned data saved successfully!")

Cleaned data saved successfully!


In [45]:
%pip install mysql-connector-python pandas

Note: you may need to restart the kernel to use updated packages.


In [23]:
import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="MySQL@2026",
    database="ecommerce_etl"
)

print("Connected to MySQL successfully!")

Connected to MySQL successfully!


In [46]:

import pandas as pd
import mysql.connector
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

clickstream_df = pd.read_csv(
    r"D:\ECommerce_ETL_Project\output\clickstream_clean.csv"
)

sales_df = pd.read_csv(
    r"D:\ECommerce_ETL_Project\output\sales_clean.csv"
)

connection_url = URL.create(
    drivername="mysql+mysqlconnector",
    username="root",
    password="MySQL@2026",
    host="localhost",
    port=3306,
    database="ecommerce_etl"
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    print("Connected to MySQL successfully!")

clickstream_df.to_sql(
    "clickstream_clean",
    con=engine,
    if_exists="replace",
    index=False
)

sales_df.to_sql(
    "sales_clean",
    con=engine,
    if_exists="replace",
    index=False
)

print("Both cleaned datasets loaded into MySQL successfully!")
engine.dispose()

Connected to MySQL successfully!
Both cleaned datasets loaded into MySQL successfully!


In [47]:
###hourly sales summary

from pyspark.sql.functions import col, hour, sum as spark_sum, countDistinct, to_date

sales_for_summary = (
    sales_clean
    .withColumn("sale_date", to_date(col("transaction_time")))
    .withColumn("sale_hour", hour(col("transaction_time")))
    .withColumn("total_amount", col("quantity") * col("price"))
)

hourly_sales_summary = (
    sales_for_summary
    .groupBy("sale_date", "sale_hour")
    .agg(
        spark_sum("total_amount").alias("total_revenue"),
        spark_sum("quantity").alias("total_quantity"),
        countDistinct("transaction_id").alias("total_transactions")
    )
    .orderBy("sale_date", "sale_hour")
)

hourly_sales_summary.show(truncate=False)

+----------+---------+-------------+--------------+------------------+
|sale_date |sale_hour|total_revenue|total_quantity|total_transactions|
+----------+---------+-------------+--------------+------------------+
|2026-10-01|10       |3200.0       |5             |4                 |
|2026-10-01|11       |300.0        |1             |1                 |
|2026-10-01|12       |4350.0       |5             |4                 |
|2026-10-01|13       |1250.0       |2             |2                 |
|2026-10-01|14       |1600.0       |2             |2                 |
|2026-10-01|15       |1750.0       |4             |3                 |
+----------+---------+-------------+--------------+------------------+



In [48]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\ECommerce_ETL_Project")
OUTPUT_PATH = PROJECT_ROOT / "output"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

hourly_sales_summary.toPandas().to_csv(
    OUTPUT_PATH / "hourly_sales_summary.csv",
    index=False
)

print("Hourly sales summary saved successfully!")

Hourly sales summary saved successfully!


In [49]:
from sqlalchemy import create_engine, URL

connection_url = URL.create(
    drivername="mysql+mysqlconnector",
    username="root",
    password="MySQL@2026",
    host="localhost",
    port=3306,
    database="ecommerce_etl"
)

engine = create_engine(connection_url)

hourly_sales_summary.toPandas().to_sql(
    "hourly_sales_summary",
    con=engine,
    if_exists="replace",
    index=False
)

engine.dispose()

print("Hourly summary loaded into MySQL!")

Hourly summary loaded into MySQL!


In [52]:
from pathlib import Path
import mysql.connector

project = Path(r"D:\ECommerce_ETL_Project")

files = [
    project / "output" / "clickstream_clean.csv",
    project / "output" / "sales_clean.csv",
    project / "output" / "hourly_sales_summary.csv"
]

for file in files:
    print(file.name, ":", "Found" if file.exists() else "Missing")

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="MySQL@2026",
    database="ecommerce_etl"
)

cursor = conn.cursor()
cursor.execute("SHOW TABLES")

print("\nMySQL tables:")
for table in cursor.fetchall():
    print(table[0])

cursor.close()
conn.close()

clickstream_clean.csv : Found
sales_clean.csv : Found
hourly_sales_summary.csv : Found

MySQL tables:
clickstream_clean
hourly_sales_summary
sales_clean
